# Suite2 Project8 Clustering

**Dataset:** Mall Customers (200 customers)

---
Run each cell in order. All outputs are generated from real data.

In [ ]:
# Setup: imports and configuration
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plotting style
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams['figure.dpi'] = 120

# Helper: load dataset
def load_data(name):
    return pd.read_csv(f'https://raw.githubusercontent.com/ulink-ibcs/A4_ML_Projects/main/datasets/{name}')

print("✅ Setup complete!")

In [ ]:
"""Project 8: Customer Segmentation — Clustering"""
import sys, os; sys.path.insert(0, os.path.dirname(__file__))

df = load_csv('mall_customers.csv')
OUTPUT_DIR = os.path.join(BASE_DIR, 'suite2_project8_outputs'); os.makedirs(OUTPUT_DIR, exist_ok=True)

## PROJECT 8: CUSTOMER SEGMENTATION — CLUSTERING

In [ ]:
print(f"Dataset: Mall Customers ({df.shape[0]} rows)")
print(df.head().to_string())

# Features for clustering
features = ['AnnualIncome_k', 'SpendingScore']
X = df[features].copy()
X_scaled = StandardScaler().fit_transform(X)

# ── 1. K-Means Clustering ──

## 1. K-Means Clustering

In [ ]:
# Elbow method
inertias = []
silhouettes = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_scaled, labels)
    silhouettes.append(sil)
    print(f"  k={k:2d}: Inertia={km.inertia_:.0f}, Silhouette={sil:.4f}")

# Elbow + Silhouette plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(K_range, inertias, 'o-', color='#2e86de', linewidth=2)
axes[0].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[0].set_ylabel('Inertia', fontsize=12)
axes[0].set_title('Elbow Method', fontsize=13)
axes[0].grid(True, alpha=0.3)

axes[1].plot(K_range, silhouettes, 'o-', color='#e74c3c', linewidth=2)
axes[1].axhline(y=max(silhouettes), color='green', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('Silhouette Analysis', fontsize=13)
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
# 'p8_elbow_silhouette')
plt.show()
print("[Saved] p8_elbow_silhouette.png")

# Best K-Means (k=5 from elbow)
best_k = 5
km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df['KMeans_Cluster'] = km.fit_predict(X_scaled)
print(f"\n=== K-Means (k={best_k}) Cluster Centers (original scale) ===")
centers = pd.DataFrame(km.cluster_centers_, columns=['AnnualIncome (scaled)', 'Spending (scaled)'])
print(centers.round(3).to_string())

# Visualize clusters
fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(df['AnnualIncome_k'], df['SpendingScore'], 
                     c=df['KMeans_Cluster'], cmap='viridis', s=60, alpha=0.7, edgecolors='black', linewidth=0.5)
centers_orig = scaler = StandardScaler().fit(X)
centers_orig = pd.DataFrame(km.cluster_centers_, columns=features)
centers_orig.iloc[:, 0] = centers_orig.iloc[:, 0] * X.std().AnnualIncome_k + X.mean().AnnualIncome_k
centers_orig.iloc[:, 1] = centers_orig.iloc[:, 1] * X.std().SpendingScore + X.mean().SpendingScore
ax.scatter(centers_orig['AnnualIncome_k'], centers_orig['SpendingScore'], 
           c='red', marker='X', s=200, edgecolors='black', linewidth=2, label='Centroids')
ax.set_xlabel('Annual Income ($K)', fontsize=12)
ax.set_ylabel('Spending Score (1-100)', fontsize=12)
ax.set_title(f'K-Means Clustering (k={best_k}) — Customer Segments', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax, label='Cluster')
plt.tight_layout()
# 'p8_kmeans_clusters')
plt.show()
print("[Saved] p8_kmeans_clusters.png")

# Cluster profiles

## 2. Cluster Profiles

In [ ]:
for c in sorted(df['KMeans_Cluster'].unique()):
    subset = df[df['KMeans_Cluster'] == c]
    print(f"\nCluster {c} (n={len(subset)}):")
    print(f"  Income:   ${subset['AnnualIncome_k'].mean():.0f}K ± {subset['AnnualIncome_k'].std():.0f}K")
    print(f"  Spending: {subset['SpendingScore'].mean():.0f} ± {subset['SpendingScore'].std():.0f}")
    print(f"  Age:      {subset['Age'].mean():.0f} ± {subset['Age'].std():.0f}")
    print(f"  Gender:   {subset['Gender'].value_counts().to_dict()}")

# ── 2. Hierarchical Clustering ──

## 3. Hierarchical Clustering (Agglomerative)

In [ ]:
hc = AgglomerativeClustering(n_clusters=best_k, linkage='ward')
df['HCluster'] = hc.fit_predict(X_scaled)

# Dendrogram
fig, ax = plt.subplots(figsize=(12, 6))
Z = linkage(X_scaled, method='ward')
dn = dendrogram(Z, truncate_mode='lastp', p=30, 
                leaf_rotation=45, leaf_font_size=10,
                color_threshold=5, above_threshold_color='gray')
ax.set_title('Dendrogram — Hierarchical Clustering (truncated)', fontsize=14)
ax.set_xlabel('Sample Index', fontsize=12)
ax.set_ylabel('Distance', fontsize=12)
plt.tight_layout()
# 'p8_dendrogram')
plt.show()
print("[Saved] p8_dendrogram.png")

# ── 3. DBSCAN ──

## 4. DBSCAN — Density-Based Clustering

In [ ]:
for eps in [0.3, 0.5, 0.7]:
    db = DBSCAN(eps=eps, min_samples=5)
    labels = db.fit_predict(X_scaled)
    n_clusters_db = len(set(labels) - {-1})
    n_noise = sum(labels == -1)
    print(f"  eps={eps:.1f}: Clusters={n_clusters_db}, Noise={n_noise} ({n_noise/len(labels):.1%})")

# Best DBSCAN
db = DBSCAN(eps=0.5, min_samples=5)
df['DBSCAN_Cluster'] = db.fit_predict(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(df['AnnualIncome_k'], df['SpendingScore'], c=df['HCluster'], cmap='viridis', s=50, alpha=0.7)
axes[0].set_title('Hierarchical Clustering', fontsize=13)
axes[0].set_xlabel('Annual Income ($K)'); axes[0].set_ylabel('Spending Score')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(df['AnnualIncome_k'], df['SpendingScore'], c=df['DBSCAN_Cluster'], cmap='viridis', s=50, alpha=0.7)
axes[1].set_title('DBSCAN Clustering', fontsize=13)
axes[1].set_xlabel('Annual Income ($K)'); axes[1].set_ylabel('Spending Score')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
# 'p8_clustering_comparison')
plt.show()
print("[Saved] p8_clustering_comparison.png")

results = {
    'elbow': {'k': list(K_range), 'inertia': inertias, 'silhouette': silhouettes},
    'best_k': best_k,
    'kmeans_score': float(silhouette_score(X_scaled, df['KMeans_Cluster'])),
    'hierarchical_score': float(silhouette_score(X_scaled, df['HCluster'])),
    'dbscan_clusters': int(len(set(df['DBSCAN_Cluster']) - {-1})),
    'dbscan_noise': int(sum(df['DBSCAN_Cluster'] == -1)),
}
    json.dump(results, f, indent=2, default=str)

print("Done.")